In [17]:
from pathlib import Path
from liffile import LifFile
import numpy as np
import tifffile
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import curve_fit
from scipy.signal import find_peaks


def find_z_crop_bounds(arr_t0, remaining_dims, n_sigma=2.0):
    """Compute Z crop bounds from the t=0 channel-0 intensity profile.

    arr_t0  : array of shape (Z, *remaining) — the t=0 slice after transposing
    remaining_dims : list of dim names corresponding to axes after Z
    n_sigma : how many Gaussian sigma either side of the peak to keep

    Returns (z_start, z_end, raw_profile, smoothed_profile, popt)
    where popt = [amplitude, mu, sigma].
    """
    # Extract channel 0 for intensity profiling
    if "C" in remaining_dims:
        c_axis = 1 + remaining_dims.index("C")   # +1 for the leading Z axis
        t0_ch0 = np.take(arr_t0, 0, axis=c_axis)  # → (Z, Y, X)
    else:
        t0_ch0 = arr_t0  # already (Z, Y, X)

    n_z = t0_ch0.shape[0]
    profile = t0_ch0.sum(axis=(-1, -2)).astype(float)
    z_idx = np.arange(n_z)

    smoothed = gaussian_filter1d(profile, sigma=2)

    # Find the highest peak (ignore secondary bumps)
    peaks, _ = find_peaks(smoothed, prominence=smoothed.max() * 0.05)
    main_peak = int(peaks[np.argmax(smoothed[peaks])]) if len(peaks) > 0 else int(np.argmax(smoothed))

    # Gaussian fit around the dominant peak
    window = max(n_z // 4, 5)
    lo, hi = max(0, main_peak - window), min(n_z, main_peak + window)

    def _gauss(x, amp, mu, sig):
        return amp * np.exp(-0.5 * ((x - mu) / sig) ** 2)

    try:
        popt, _ = curve_fit(
            _gauss, z_idx[lo:hi], smoothed[lo:hi],
            p0=[smoothed[main_peak], float(main_peak), n_z / 6],
            maxfev=5000,
        )
        mu, sigma = popt[1], abs(popt[2])
        popt = [popt[0], mu, sigma]
    except Exception:
        # Fallback: FWHM estimate
        mu = float(main_peak)
        half_max = smoothed[main_peak] / 2 if smoothed[main_peak] > 0 else 1.0
        left  = next((i for i in range(main_peak, -1, -1) if smoothed[i] < half_max), 0)
        right = next((i for i in range(main_peak, n_z)    if smoothed[i] < half_max), n_z - 1)
        sigma = max((right - left) / 2.355, 1.0)
        popt = [float(smoothed[main_peak]), mu, sigma]

    z_start = max(0,   int(np.round(mu - n_sigma * sigma)))
    z_end   = min(n_z, int(np.round(mu + n_sigma * sigma)))
    return z_start, z_end, profile, smoothed, popt

In [18]:
lif_paths   = Path(r"Z:\Bel\Avalon_Cropping\Original")  # <-- update this
output_dir = Path(r"Z:\Bel\Avalon_Cropping")

print(lif_paths)

Found 4 LIF files:
  20260417_d5_Bloom_Donor2FB_1to12_permeability_texasred70kDa.lif
  20260417_d5_Bloom_TNF_10ngml_Donor2FB_1to12_permeability_texasred70kDa.lif
  20260417_d5_Iris_Donor2FB_1to12_permeability_texasred70kDa.lif
  20260417_d5_Iris_TNF_10ngml_Donor2FB_1to12_permeability_texasred70kDa.lif


In [22]:



lif_path = lif_paths[0]
print(lif_path)
N_SIGMA  = 2.0   # keep µ ± N_SIGMA*σ of the central Gaussian — adjust after previewing in cell 3
lif_stem = lif_path.stem.replace("/", "_")
for lif_path in lif_paths:
    with LifFile(lif_path) as lif:
        for img in lif.images:
            name = "".join(img.path).replace("/", "_")
            dims = tuple(img.dims)

            try:
                data = img.asarray()
            except (KeyError, IndexError) as e:
                print(f"\n{name}: SKIP - could not read image data ({type(e).__name__}: {e})")
                continue

            print(f"\n{name}: dims={dims}, shape={data.shape}")

            if "T" not in dims or "Z" not in dims:
                print(f"  SKIP - missing T or Z axis")
                continue

            t_ax = dims.index("T")
            z_ax = dims.index("Z")
            n_t  = data.shape[t_ax]
            n_z  = data.shape[z_ax]

            if n_t != 3:
                print(f"  SKIP - expected 3 timepoints, got {n_t}")
                continue

            # Move T to axis-0 and Z to axis-1 for easy slicing
            axes_order = list(range(data.ndim))
            axes_order.remove(t_ax)
            axes_order.remove(z_ax)
            axes_order = [t_ax, z_ax] + axes_order
            arr = np.transpose(data, axes_order)
            # arr shape: (T, Z, *remaining)

            remaining_dims = [dims[i] for i in axes_order[2:]]
            print(f"  Z slices = {n_z}, remaining dims = {remaining_dims}")

            # Determine crop bounds from t=0 channel-0 intensity profile only
            z_start, z_end, _, _, popt = find_z_crop_bounds(arr[0], remaining_dims, n_sigma=N_SIGMA)
            n_out = z_end - z_start
            if n_out <= 0:
                print(f"  SKIP - gaussian crop produced 0 slices")
                continue
            print(f"  Gaussian µ={popt[1]:.1f}, σ={popt[2]:.1f} → crop slices {z_start}–{z_end} ({n_out} kept)")

            # Apply the same crop to ALL timepoints
            cropped = arr[:, z_start:z_end, ...]
            print(f"  Cropped shape (T, Z, ...): {cropped.shape}")

            # Ensure TZCYX order for ImageJ hyperstack
            has_c = "C" in remaining_dims
            if not has_c:
                # remaining = [Y, X] → insert a C dimension
                cropped = cropped[:, :, np.newaxis, :, :]

            # Read pixel sizes from LIF metadata for ImageJ calibration
            metadata   = {}
            resolution = None
            try:
                xa = img.asxarray()
                coords = xa.coords
                if "X" in coords and coords["X"].size >= 2:
                    x_um = abs(float(coords["X"][1] - coords["X"][0])) * 1e6
                    resolution = (1.0 / x_um, 1.0 / x_um)
                    metadata["unit"] = "um"
                    print(f"  XY pixel size = {x_um:.4f} µm")
                if "Z" in coords and coords["Z"].size >= 2:
                    z_um = abs(float(coords["Z"][1] - coords["Z"][0])) * 1e6
                    metadata["spacing"] = z_um
                    print(f"  Z step = {z_um:.4f} µm")
            except Exception as e:
                print(f"  [WARN] Could not read pixel sizes: {e}")

            out_path = output_dir / f"{lif_stem}__{name}_cropped.tif"
            tifffile.imwrite(
                str(out_path),
                cropped.astype(data.dtype),
                imagej=True,
                resolution=resolution,
                metadata=metadata,
            )
            print(f"  Saved -> {out_path}")

print("\nDone.")

Z:\Bel\Avalon_Cropping\Original\20260417_d5_Bloom_Donor2FB_1to12_permeability_texasred70kDa.lif

TileScan 3_P 1: dims=('T', 'C', 'Z', 'Y', 'X'), shape=(3, 2, 31, 512, 512)
  Z slices = 31, remaining dims = ['C', 'Y', 'X']
  Gaussian µ=8.8, σ=4.4 → crop slices 0–18 (18 kept)
  Cropped shape (T, Z, ...): (3, 18, 2, 512, 512)
  XY pixel size = 2.2750 µm
  Z step = 5.0001 µm
  Saved -> Z:\Bel\Avalon_Cropping\20260417_d5_Bloom_Donor2FB_1to12_permeability_texasred70kDa__TileScan 3_P 1_cropped.tif

TileScan 3_P 2: dims=('T', 'C', 'Z', 'Y', 'X'), shape=(3, 2, 31, 512, 512)
  Z slices = 31, remaining dims = ['C', 'Y', 'X']
  Gaussian µ=8.6, σ=4.1 → crop slices 0–17 (17 kept)
  Cropped shape (T, Z, ...): (3, 17, 2, 512, 512)
  XY pixel size = 2.2750 µm
  Z step = 5.0001 µm
  Saved -> Z:\Bel\Avalon_Cropping\20260417_d5_Bloom_Donor2FB_1to12_permeability_texasred70kDa__TileScan 3_P 2_cropped.tif

TileScan 3_P 3: dims=('T', 'C', 'Z', 'Y', 'X'), shape=(3, 2, 31, 512, 512)
  Z slices = 31, remaining d